# Evaluation produit du RAG multimodal

Ce notebook evalue chaque etape avec des qrels humains. Le nombre final d'images est variable : la taille de lot du VLM ne limite jamais le nombre de resultats pertinents.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(path for path in candidates if (path / 'ml').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from ml.evaluation import (
    binary_classification_metrics,
    evaluate_retrieval_run,
)

print(f'Project root: {ROOT}')

## 1. Qrels et sorties par etape

Remplacez ces identifiants par ceux de vos photos. Grade 0 = incorrect, 1 = pertinent, 2 = correspondance directe. Ajoutez aussi des requetes sans resultat attendu.

In [ ]:
qrels = {
    'food_table': {'food_exact': 2, 'food_ok': 1},
    'people_playing': {'football': 2, 'children_game': 1},
    'no_match': {},
}

runs = {
    'food_table': {
        'lexical': ['food_exact', 'bedroom_table', 'mug_table'],
        'text_image': ['food_ok', 'food_exact', 'mug_table'],
        'rrf': ['food_exact', 'food_ok', 'mug_table', 'bedroom_table'],
        'judge': ['food_exact', 'food_ok'],
        'final': ['food_exact', 'food_ok'],
    },
    'people_playing': {
        'lexical': ['football', 'children_game', 'animal'],
        'text_image': ['animal', 'football'],
        'rrf': ['football', 'animal', 'children_game'],
        'judge': ['football', 'children_game'],
        'final': ['football', 'children_game'],
    },
    'no_match': {
        'lexical': [],
        'text_image': ['near_but_wrong'],
        'rrf': ['near_but_wrong'],
        'judge': [],
        'final': [],
    },
}

In [ ]:
report = evaluate_retrieval_run(qrels, runs, cutoffs=(1, 5, 10, 20, 50))
pd.Series(report['aggregate']['ranking']).to_frame('score')

In [ ]:
stage_recall = pd.Series(report['aggregate']['stage_recall'])
ax = stage_recall.plot(kind='bar', ylim=(0, 1), color='#2f6f62')
ax.set_title('Rappel moyen conserve par etape')
ax.set_ylabel('Recall')
ax.grid(axis='y', alpha=0.25)
plt.show()

pd.DataFrame({
    query_id: values['final_set']
    for query_id, values in report['queries'].items()
}).T

## 2. Calibration du juge VLM

Les labels doivent venir d'une verification humaine des pixels. Les scores ci-dessous sont des probabilites de pertinence : pour la sortie du VLM, utilisez confidence si relevant=true et 1-confidence sinon. Testez plusieurs seuils sur le jeu de calibration, puis conservez le seuil choisi pour le jeu de test gele.

In [ ]:
human_labels = [1, 1, 1, 0, 0, 0]
vlm_confidences = [0.96, 0.82, 0.61, 0.79, 0.32, 0.08]

threshold_rows = []
for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    threshold_rows.append(
        binary_classification_metrics(
            human_labels,
            vlm_confidences,
            threshold=threshold,
        )
    )

calibration = pd.DataFrame(threshold_rows).set_index('threshold')
calibration[['precision', 'recall', 'f1', 'false_positive_rate', 'false_negative_rate', 'brier_score', 'expected_calibration_error']]

In [ ]:
ax = calibration[['precision', 'recall', 'f1']].plot(marker='o')
ax.set_ylim(0, 1.05)
ax.set_title('Compromis precision-rappel du juge')
ax.set_xlabel('Seuil de confiance')
ax.grid(alpha=0.25)
plt.show()

## 3. Lecture des echecs

- Si le rappel chute avant RRF, augmenter ou ameliorer les canaux de rappel.
- Si le rappel chute au juge, inspecter les faux negatifs et recalibrer le seuil/prompt.
- Si la precision finale est faible, ajouter des hard negatives visuels au jeu d'evaluation.
- Mesurer P50/P95/P99, RAM et VRAM separement pour TinyCLIP, Qdrant, RRF et VLM.
- Ne jamais choisir un modele sur les labels derives de ses propres captions.